Abbiamo parlato di BM25, ora vediamo l'altra faccia della medaglia: i Language Models (LM).

I **Language Models** sono modelli probabilistici proprio come BM25, ma rispondono a una domanda molto diversa. Mentre in BM25 la domanda è "Quanto è rilevante questo documento per questa query?", nei Language Models si vuole invece rispondere a qualcosa di apparentemente folle: **"Quanto è probabile osservare proprio quella a query a partire da un certo documento?"**. Più tale probabilità è alta, più il documento è rilevante per la query. 

("folle" perché i documenti non sono fatti per generare query, ma per dare informazioni)

Ci eravamo affidati a Bayes per ottenere tale formula, così come avevamo fatto per BM25 (ma avevamo messo a sx $d$, con LM mettiamo a sx $q$):
  $$\Pr(R \mid d,q)=\frac{\Pr(q \mid R,d)\cdot \Pr(R \mid d)}{\Pr(q \mid d)}$$
In realtà negli LM non faremo affidamento ad R (la possibilità che il documento sia rilevante), ma partiremo direttamente da $\Pr(q \mid d)$, che è proprio la probabilità di osservare la query a partire dal documento.

### Idea di base

Per capire il funzionamento di LM, introduciamo la quantità $\Pr(q \mid M_d)$, dove $M_d$ è il language model associato al documento $d$. Questa probabilità rappresenta proprio la probabilità di osservare la query $q$ a partire dal modello linguistico del documento $d$: **un documento sarà considerato più rilevante per una query se il suo modello linguistico "rende naturale" la generazione di quella query**.

Intuitivamente infatti, se un utente cerca $q=\text{Frodo Ring}$, allora un documento che parla di "Frodo", "Sam", "Ring", "Mordor"... dovrebbe avere alta probabilità di "produrre" quella query, mentre al contrario un documento che parla di cucina o calcio dovrebbe avere probabilità molto più bassa.

L'idea allora è che **ogni documento definisce una distribuzione di probabilità sui termini**: ogni documento $d$ sarà trasformato in un language model $M_d$ che assegna una probabilità a ogni termine del vocabolario. 

Tornando all'esempio quindi un documento su Tolkien avrà $\Pr(\text{Frodo} \mid M_d)$ alta, mentre un documento su cucina avrà $\Pr(\text{Frodo} \mid M_d)$ bassa. 

Una volta ottenute tutte queste probabilità, potremo quindi calcolare $\Pr(q \mid M_d)$ e ordinare i documenti in base a questo valore.

**Pipeline generale dei LM in IR**:
1. Definire un language model da utilizzare (ad esempio un unigram model).
2. Ogni documento $d$ diventa un language model generativo $M_d$.
3. Stimare per ogni documento e per ogni termine nella query la probabilità $\Pr(t \mid M_d)$.
4. Fare Smoothing (infatti se un termine $t$ nella query non appare nel documento -> $\Pr(t \mid M_d) = 0$ e ciò vedremo che annulla tutta la probabilità $\Pr(q \mid M_d)$).
5. Calcolare $\Pr(q \mid M_d)$ e ordinare i documenti in base a questo valore.

### Language Models: definizione generale
Prima di capire quale language model utilizzeremo, è fondamentale capire cos'è in generale un language model.

**L'idea generale di un Language Model è quella di modellare probabilisticamente la generazione di un testo**. Si vuole cioè generare del testo una parola alla volta, scegliendo di volta in volta il termine successivo in base a una distribuzione di probabilità che dipende dalle parole già generate.

Formalmente quindi un LM assegna probabilità del tipo $$\Pr(w_i \mid w_1, w_2, ..., w_{i-1})$$ ossia probabilità di osservare la parola $w_i$ dato tutto ciò che è stato generato fino a quel punto.

Questa dipendenza rispetto al contesto passato è il principio alla base di modelli complessi come Markov Models o LLMs più moderni come GPT.

In questo senso un LM può essere visto come un automa a stati finiti, dove:
- ogni stato rappresenta una sequenza di parole già generate (contesto corrente)
- le transizioni rappresentano la probabilità di generare una parola successiva dato il contesto corrente.

Man mano che si procede nei vari stati dell'automa (e quindi nella costruzione della frase), **si può calcolare la probabilità complessiva di generare la frase fino a quel punto moltiplicando le probabilità di ogni parola generata** (chain rule: $\Pr(w_1, w_2, ..., w_n) = \prod_{i=1}^n \Pr(w_i \mid w_1, w_2, ..., w_{i-1})$).

Ognuna di queste probabilità è però della forma $\Pr(w_i \mid w_1, w_2, ..., w_{i-1})$, ossia ogni passaggio di stato dipende da tutto ciò che è stato generato fino a quel punto.

#### Unigram Language Model
In IR si utilizza però un Language Model molto più semplice, che evita del tutto questa dipendenza dal contesto passato: **l'Unigram Language Model**.

Nel Retrieval classico infatti si fa una **fortissima assunzione di indipendenza**:
$$
\Pr(w_i \mid w_1, w_2, ..., w_{i-1}, M_d) = \Pr(w_i \mid M_d)
$$
ossia la probabilità di osservare la parola \(w_i\) non dipende più dalla storia precedente, ma soltanto dal language model del documento \(M_d\).ossia **la probabilità di una parola NON dipende dalla storia delle parole già generate, ma soltanto dal documento $d$, cioè dal suo language model $M_d$**.

In questo modo ogni documento diventa semplicemente, secondo questo language model, una distribuzione di probabilità sui termini del vocabolario:
$$M_d = \{\Pr(t \mid M_d): t \in V\}$$
dove $V$ è il vocabolario.

Conseguenza fondamentale in questo senso è che per un qualsiasi documento, la probabilità di generare una query "Frodo Sam saw" è la stessa di generare "Sam saw Frodo": si perde completamente l'ordine e conta solo la probabilità dei singoli termini.

Ma perché utilizzare un modello così semplice? Perché nell'IR **non vogliamo generare testo realistico, ma solo fare ranking dei documenti**. Quindi usare un modello così semplice è molto vantaggioso poiché stimare probabilità di unigrammi è più facile, portando a un modello molto efficiente.

Il calcolo di $\Pr(q \mid M_d)$ consisterà nella **moltiplicazione** delle probabilità di ogni termine presente nella query secondo il modello del documento.

Nell'immagine sotto si vede proprio l'applicazione di questo principio rispetto a due documenti diversi, ognuno con il proprio unigram language model (e quindi con probabilità diverse per ogni termine). Facendo il calcolo rispetto alla query si arriva alla conclusione che il secondo documento è più rilevante per la query, poiché "più propenso" a generarla secondo le distribuzuioni di probabilità dei suoi termini.

<img src="img/coo.png" width="400"/>

(STOP è presente solo nell’esempio didattico per mostrare la generazione di una sequenza completa. Nella query likelihood per IR, di solito si calcolano solo le probabilità dei termini effettivi della query, senza includere STOP.)

## Applicazione
L'obiettivo finale è ordinare i documenti rispetto a una query $q$. Sfruttando Bayes:
$$ 
\Pr(d \mid q) = \frac{\Pr(q \mid d) \cdot \Pr(d)}{\Pr(q)}$$ 
dove $\Pr(d)$ è la probabilità a priori di un documento, $\Pr(q)$ è la probabilità a priori di una query e $\Pr(q \mid d)$ è la probabilità di osservare la query a partire dal documento $d$.

Come già detto nei Language Models un documento viene rappresentato tramite il suo language model $M_d$, quindi si fa l'approssimazione $\Pr(q \mid d) \approx \Pr(q \mid M_d)$.

Volendo fare ranking inoltre **la query è fissata rispetto all'ordinamento dei documenti** -> $\Pr(q)$ è uguale per tutti i documenti, quindi non influenza l'ordinamento -> si può ignorare.

**L'ultima assunzione importante è che si assume che tutti i documenti siano ugualmente probabili a priori** -> $\Pr(d)$ è uguale per tutti i documenti, quindi non influenza l'ordinamento -> si può ignorare.

Quindi il ranking finale dipenderà solo da $\Pr(q \mid M_d)$, questa quantità è nota come **query likelihood**.

Dopodiché si applica la semplificazione vista prima relativamente all'**Unigram Language Model**; normalmente un language model dipenderebbe da tutto ciò che è stato generato fino a quel punto, ma in IR si assume totale indipendenza rispetto al contesto passato, quindi si arriva alla formula finale:
$$\Pr(q \mid M_d) = \prod_{w_i}^{|q|} \Pr(w_i \mid M_d)$$
si ricorda ancora una volta che in questo modo la query è trasformata in una bag-of-words, nel senso che "Frodo Ring" e "Ring Frodo" hanno la stessa probabilità di essere generate a partire da un documento, poiché si perde completamente la storia e quindi l'ordine dei termini. **Per questo motivo conta soltanto quante volte compare ogni termine nella query, non l'ordine in cui compaiono**. Definiamo $tf_{t_i, q}$ come il numero di volte che il termine $t_i$ compare nella query $q$.

Allora, poiché l'ordine non conta, possiamo vedere la query come un campionamento multinomiale dei termini dal language model del documento ([vedi parentesi multinomiale](#parentesi-multinomiale)) e quindi: 
$$\Pr(q \mid M_d) = \frac{|q|!}{\prod_{t \in V} tf_{t, q}!} \cdot \prod_{t \in V} \Pr(t \mid M_d)^{tf_{t, q}}$$
il termine a sinistra conta il numero di modi in cui i termini della query possono essere ordinati (dato che l'ordine non conta) mentre la parte a destra ci dice che se un termine compare molte volte nella query, allora la sua probabilità contribuisce maggiormente alla likelihood della query.

Dal momento che stiamo facendo ranking, il termine a sinistra è uguale per tutti i documenti (dato che la query è fissa) e quindi non influenza l'ordinamento -> si può ignorare. Otteniamo quindi la formula finale, utilizzata davvero per il ranking:

$$\Pr(q \mid M_d) \approx \prod_{t: tf_{t, q} > 0} \Pr(t \mid M_d)^{tf_{t, q}}$$

**Interpretazione intuitiva: un documento riceve score alto se assegna alta probabilità ai termini presenti nella query, e se questi termini compaiono molte volte nella query**.

Si formalizza poi il language model come segue:
$$M_d = [\Pr(t_1 \mid M_d), \Pr(t_2 \mid M_d), \ldots, \Pr(t_n \mid M_d)]$$
ossia ogni documento è rappresentato come un vettore di probabilità, una per ogni termine del vocabolario. Poiché $M_d$ definisce una distribuzione di probabilità sui termini del vocabolario, allora deve valere il vincolo:
$$\sum_{i=1}^{|V|} \Pr(t_i \mid M_d) = 1$$

A questo punto ci si chiede come stimare $\Pr(t_i \mid M_d)$. Il metodo più semplice è quello di utilizzare la **Maximum Likelihood Estimation (MLE)**, la stima esce:
$$\hat{P}(t_i \mid M_d) = \frac{tf_{t_i, d}}{|d|}$$
(banalmente, la probabilità di osservare un termine a partire da un documento è data dalla frequenza del termine nel documento diviso la lunghezza del documento).

La stima MLE però ha un problema enorme: se un termine della query non compare nel documento, allora $tf_{t_i, d} = 0$ e quindi $\Pr(t_i \mid M_d) = 0$. Di conseguenza, poiché la query likelihood è data dal prodotto delle probabilità dei termini, si annullerebbe completamente la probabilità di osservare la query a partire da quel documento, anche se magari il documento è molto rilevante per la query (ad esempio se la query è "Frodo Ring" e il documento parla di "Frodo", "Sam", "Mordor" ma non di "Ring", allora secondo MLE quel documento avrebbe probabilità zero di generare la query, anche se in realtà è molto rilevante). Questo problema è noto come **veto power** (un singolo termine assente ha il potere di eliminare completamente un documento dal ranking).

Per risolvere il problema è molto importante quindi applicare **smoothing**.

### Laplace Smoothing
Il metodo più semplice per fare smoothing è il **Laplace Smoothing** (o add-one smoothing). L'idea è quella di aggiungere 1 a tutte le frequenze dei termini, in questo modo anche i termini assenti avranno una frequenza di almeno 1 e quindi una probabilità maggiore di zero. La formula diventa:
$$P_{Lap}(t \mid M_d) = \frac{tf_{t, d} + 1}{|d| + |V|}$$
dove al denominatore si aggiunge $|V|$ per mantenere la normalizzazione della distribuzione di probabilità (dato che stiamo aggiungendo 1 a ogni termine del vocabolario, stiamo aggiungendo un totale di $|V|$ al conteggio totale dei termini).

Laplace smoothing funziona matematicamente, ma ha un problema: aggiungere 1 a tutti i termini del vocabolario, anche a quelli che non compaiono mai nel documento, è un'operazione troppo aggressiva. 

es. documento $|d| = 100$, vocabolario $|V| = 10000$: senza smoothing, se "Frodo" compare 5 volte -> $\Pr(\text{Frodo} \mid M_d) = 5/100 = 0.05$; con Laplace smoothing -> $P_{Lap}(\text{Frodo} \mid M_d) = (5+1)/(100+10000) = 6/10100 \approx 0.000594$, una probabilità molto più bassa, che non riflette adeguatamente la rilevanza del documento rispetto alla query. Questo succede perché aggiungendo 1 a tutti i 10k termini del vocavolario è come se avessimo aggiunto 10k termini artificiali al documento.

Quindi Laplace Smoothing più didattico che altro, non usato in pratica.

### Jelinek-Mercer Smoothing
Invece di "inventare" artificialmente una parola per ogni termine del vocabolario, usiamo la statistica dell'intera collezione.

Definiamo anzitutto il **collection Language Model** $M_c$ come la distribuzione di probabilità sui termini del vocabolario stimata a partire da tutta la collezione di documenti. Usando di nuovo la stima MLE, otteniamo:
$$\hat{P}(t \mid M_c) = \frac{cf_t}{T}$$
dove $cf_t$ è la frequenza del termine $t$ in tutta la collezione (collection frequency) e $T = \sum_{t \in V} cf_t$ è il numero totale di token (non distinti) in tutta la collezione.

Questa quantità rappresenta quanto il termine è comune in generale -> "the" avrà probabilità molto alta, "Frodo" avrà probabilità molto più bassa.

L'idea dietro **Jelinek-Mercer** è quella di fare una media pesata tra il language model del documento e il language model della collezione:
$$P_{JM}(t \mid M_d) = \lambda \frac{tf_{t, d}}{|d|} + (1-\lambda) \frac{cf_t}{T}$$
dove la parte a sinistra è la stima MLE del documento e ci dice quanto il termine è importante per il documento specifico, mentre la parte a destra è la stima MLE della collezione e ci dice quanto il termine è comune in generale.

Jelinek-Mercer risolve chiaramente il problema degli zeri in quanto se un termine non compare nel documento resta comunque la probabilità data dalla collezione, che è sempre maggiore di zero. 

Il parametro $\lambda \in [0, 1]$ controlla il peso che diamo al documento rispetto alla collezione: se $\lambda$ è vicino a 1, diamo più peso al documento, se è vicino a 0, diamo più peso alla collezione. In un certo senso un $\lambda$ alto ci sta dicendo che serve davvero che un termine compaia nel documento per considerarlo rilevante, il comportamento assomiglia a una AND query. **Al contrario, un $\lambda$ basso dà più peso alla collezione, è utile per query lunghe e verbose**.

In generale si tende a scegliere $\lambda$ piuttosto alto per favorire l'usabilità dell'utente aspettandosi query brevi (se un utente ha cercato certe parole nella query, si aspetta sicuramente che appaiano tutte nel documento). Si tratta comunque di un parametro da ottimizzare empiricamente tramite benchmark.

Di seguito un esempio completo di applicazione di Jelinek-Mercer smoothing, con $\lambda = 0.5$:

<p>
<img src="img/jm1.png" width="33%"/>
<img src="img/jm2.png" width="33%"/>
<img src="img/jm3.png" width="33%"/>
</p>


**Esercizio** (da fare): calcolare il ranking dei documenti

<img src="img/jm4.png" width="300"/>

**Jelinek-Mercer ha però il seguente difetto: usa lo stesso valore di $\lambda$ per tutti i documenti, indipendentemente dalla loro lunghezza.** 

Ciò è problematico perché un documento lungo contiene molta informazione -> dovremmo fidarci di più del documento e quindi usare un $\lambda$ più alto, mentre un documento corto contiene poca informazione -> dovremmo fidarci di più della collezione e quindi usare un $\lambda$ più basso.

### Dirichlet Smoothing
Dirichlet mantiene la stessa filosofia di Jelinek-Mercer, ossia fare una media pesata tra il documento e la collezione, ma fa sì che la quantità di smoothing (quindi il peso dato alla collezione) dipenda dalla lunghezza del documento.

Invece di aggiungere direttamente $\hat{P}(t \mid M_c) = \frac{cf_t}{T}$ come probabilità da interpolare (come fa Jelinek-Mercer), Dirichlet la interpreta come **pseudo-count**.

In pratica Dirichlet aggiunge delle occorrenze fittizie di ogni termine del vocabolario al documento in proporzione alla loro probabilità nella collezione, secondo un nuovo parametro $\mu$ che controlla quante di queste occorrenze fittizie aggiungere (e quindi quanto smoothing applicare): il numero di queste occorrenze fittizie sarà dato da $\mu \hat{P}(t \mid M_c)$

es. supponiamo $\hat{P}(\text{Ring} \mid M_c) = 0.01$ e $\mu = 100$, allora il numero di occorrenze fittizie di "Ring" che aggiungiamo ad ogni documento è $\mu \hat{P}(\text{Ring} \mid M_c) = 100 \cdot 0.01 = 1$ (è come se Ring comparisse virtualmente almeno una volta in ogni documento, anche se in realtà non dovesse comparirci).

Il **conteggio reale** di un termine $t$ in un documento $d$ resta comunque $tf_{t, d}$, quindi il numero di occorrenze di $t$ in totale dentro $d$ sarà la sua somma con le occorrenze fittizie, ossia $tf_{t, d} + \mu \hat{P}(t \mid M_c)$ (conteggi reali + conteggi virtuali). 

Per ottenere una distribuzione di probabilità però come al solito dobbiamo normalizare, otteniamo quindi:
$$P_{Dir}(t \mid M_d) = \frac{tf_{t, d} + \mu \hat{P}(t \mid M_c)}{|d| + \mu}$$
dove al denominatore $|d|$ perché i conteggi reali sommano a $|d|$ e $\mu$ perché i conteggi virtuali sommano a $\mu$ (dato che stiamo aggiungendo $\mu \hat{P}(t \mid M_c)$ occorrenze fittizie per ogni termine del vocabolario, stiamo aggiungendo un totale di $\mu$ occorrenze fittizie al documento dato che $M_c$ è una distribuzione di probabilità e quindi le varie $\hat{P}(t \mid M_c)$ sommano a 1).

Per capire perché Dirichlet fa smoothing in base alla lunghezza del documento, facciamo alcuni passaggi algebrici:
$$P_{Dir}(t \mid M_d) = \frac{tf_{t, d} + \mu \hat{P}(t \mid M_c)}{|d| + \mu} = \frac{|d|}{|d| + \mu} \cdot \frac{tf_{t, d}}{|d|} + \frac{\mu}{|d| + \mu} \cdot \hat{P}(t \mid M_c)$$
ricordando che $\frac{tf_{t, d}}{|d|} = \hat{P}(t \mid M_d)$ otteniamo:
$$P_{Dir}(t \mid M_d) = \frac{|d|}{|d| + \mu} \cdot \hat{P}(t \mid M_d) + \frac{\mu}{|d| + \mu} \cdot \hat{P}(t \mid M_c)$$
Definendo a questo punto $\lambda_d = \frac{|d|}{|d| + \mu}$, la formula diventa:
$$P_{Dir}(t \mid M_d) = \lambda_d \hat{P}(t \mid M_d) + (1 - \lambda_d) \hat{P}(t \mid M_c)$$
Questa formula mostra quindi che **Dirichlet è in realtà un'interpolazione documento-collezione, proprio come Jelinek-Mercer, ma con un $\lambda = \frac{|d|}{|d| + \mu}$ che dipende dalla lunghezza del documento**. In particolare:
- se $|d| \gg \mu$, allora $\lambda_d \approx 1$ e quindi $P_{Dir}(t \mid M_d) \approx \hat{P}(t \mid M_d)$, ossia per documenti molto lunghi si dà più peso al documento e meno alla collezione (comportamento simile a una AND query, se un termine della query non è presente in un documento molto lungo è grave).
- se $|d| \ll \mu$, allora $\lambda_d \approx 0$ e quindi $P_{Dir}(t \mid M_d) \approx \hat{P}(t \mid M_c)$, ossia per documenti molto corti si dà più peso alla collezione e meno al documento.

Il parametro $\mu$ in Dirichlet controlla quindi la quantità di smoothing. Se in particolare $\mu$ aumenta, allora $\lambda_d$ diminuisce e quindi si dà più peso alla collezione -> più smoothing. Se invece $\mu$ diminuisce, allora $\lambda_d$ aumenta e quindi si dà più peso al documento -> meno smoothing. 

Con Dirichlet, la query likelihood finale diventa:
$$P_{Dir}(q \mid M_d) = \prod_{i=1}^{|q|} P_{Dir}(t_i \mid M_d)$$
dove $P_{Dir}(t_i \mid M_d)$ è dato dalla formula di Dirichlet vista prima.

Un problema fondamentale che a questo punto affligge in generale la query likelihood, indipendentemente dal tipo di smoothing, è che per calcolarla è necessario **moltiplicare moltissime probabilità molto piccole**. Questo porta a valori troppo piccoli per un calcolatore, che andrebbe in underflow -> per risolvere questo problema si calcola il log della query likelihood, che trasforma i prodotti in somme e quindi evita il problema dell'underflow. La formula diventa:
$$\log P_{Dir}(q \mid M_d) = \sum_{i=1}^{|q|} \log P_{Dir}(t_i \mid M_d)$$

### Conclusioni
In definitiva, sia Jelinek-Mercer che Dirichlet sono due metodi di smoothing che combinano evidenza del documento e della collezione, la differenza sta nel peso assegnato alle due componenti. 

Jelinek-Mercer utilizza un $\lambda$ fisso per tutti i documenti, mentre Dirichlet fa sì che il peso dato al documento rispetto alla collezione dipenda dalla lunghezza del documento stesso, con documenti più lunghi che ricevono più peso e documenti più corti che ricevono meno peso.

Empiricamente, è stato mostrato che:
- **Dirichlet spesso funziona molto bene per query corte/keyword queries** in quanto danno molto peso al documento
- **Jelinek-Mercer spesso funziona meglio per query lunghe e verbose** in quanto query lunghe hanno più probabilità di contenere termini mancanti, per cui uno smoothing più aggressivo (quindi più peso alla collezione) è più efficace.

Sia $\mu$ che $\lambda$ sono parametri da ottimizzare empiricamente tramite benchmark.

VSM vs. BM25 vs. LM:
- VSM: basato sulla similarità geometrica, tipicamente cosine similarity, sfrutta rappresentazioni vettoriali dei documenti. L'idea alla base è che documenti e query sono rappresentati come vettori in un iperspazio ad altissima dimensionalità
- BM25 e LM sono motivati probabilisticamente: il primo parte dal probabilistic retrieval framework (quello con R) modellando la probabilità che un documento sia rilevante data una query, il secondo modella la probabilità della query dato un documento.

**Ruolo della term frequency**:
- LM: usata direttamente per stimare la probabilità, es. $\hat{P}(t \mid M_d) = \frac{tf_{t, d}}{|d|}$
- BM25 e Vector Space: la tf viene trasformata in uno scoring factor, non inuna probabilità vera e propria

**Normalizzazione della lunghezza del documento**:
- BM25: normalizza esplicitamente la tf in base alla lunghezza del documento, con un parametro $b$ che controlla quanto peso dare alla normalizzazione
- LM: con Dirichlet, la normalizzazione è implicita nel fatto che documenti più lunghi ricevono più peso rispetto alla collezione, mentre documenti più corti ricevono meno peso.
- VSM: non ha una normalizzazione esplicita della lunghezza del documento, ma la cosine similarity normalizza implicitamente i vettori.

**Inverse Document Frequency (IDF)** (più un termine è raro, più è significativo per il documento): è usato esplicitamente in BM25 e VSM, mentre nei **non esiste idf esplicito**.

Questo è il più grande "limite" di LM rispetto a BM25 e VSM: non essendo presente esplicitamente un fattore di tipo idf, ogni termine della query contribuisce alla probabilità $\Pr(q \mid M_d)$ senza tenere conto di quanto sia raro o comune quel termine nella collezione -> un termine molto comune come "the" può influire molto sullo score dei documenti. **In realtà però, dal momento che la probabilità di un termine nella collezione viene considerata allo stesso modo nel calcolo dello score per tutti i documenti, il ranking finale non sarà influenzato da questo problema!**

LM vantaggioso rispetto a BM25 e VSM per la sua semplicità: BM25 arriva ad una formula finale molto complessa dopo numerosissime assunzioni e semplificazioni, VSM ragionamenti geometrici del tutto empirici. LM invece arriva a una formula finale molto semplice dopo ragionamenti molto più lineari.

Ultima osservazione importante: BM25 e VSM ragionano molto su document frequency, mentre gli LM ragionano molto anche sulla collection frequency.

A questo punto l'ultimo problema che ci rimane è che per calcolare lo score dobbiamo considerare tutti i documenti che contengano almeno una parola della query (OR -> moltissimi documenti), dobbiamo studiare delle tecniche per ottimizzare in questo senso

## Parentesi Multinomiale
La multinomiale rappresenta la generalizzazione della distribuzione binomiale quando le possibili categorie sono più di due. 

Es. dado a 6 facce: la distribuzione multinomiale ci dice qual è la probabilità di ottenere una certa combinazione di risultati (ad esempio 2 "1", 3 "2" e 1 "3") in un certo numero di lanci. In pratica conta quante volte esce ciascun risultato, dato un certo numero di estrazioni.

Nel caso dei Language model invece del dado abbiamo il vocabolario di termini $V = \{\text{Frodo}, \text{Ring}, \text{Sam}, ...\}$ e per ogni termine il language model del documento assegna una probabilità $\Pr(\text{Frodo} \mid M_d)$, $\Pr(\text{Ring} \mid M_d)$, $\Pr(\text{Sam} \mid M_d)$, ...

Quando generiamo una query lunga $|q|$, è come se facessimo $|q|$ estrazioni di questa distribuzione. Se la query è per esempio $q=\text{Frodo Ring Frodo}$, allora non ci interessa l'ordine (perché sappiamo di lavorare con unigram) ma i conteggi $tf_{\text{Frodo}, q} = 2$ e $tf_{\text{Ring}, q} = 1$.

La multinomiale serve a calcolare la probabilità di ottenere proprio quei conteggi di parole. La formula è la seguente:
$$\Pr(q \mid M_d) = \frac{|q|!}{\prod_{t \in V} tf_{t, q}!} \cdot \prod_{t \in V} \Pr(t \mid M_d)^{tf_{t, q}}$$

(la parte a dx ci dice: moltiplico le probabilità delle parole ripetendole quante volte compaiono nella query, mentre la parte a sx conta in quanti modi posso ordinare le parole della query, dato che l'ordine non conta)

**Vedi esempio Dirichlet nelle slide**